# smolagents — code as the action

Most frameworks have the agent emit a JSON tool call. **smolagents** has it emit Python code — the framework runs it in a sandbox and feeds the result back. Code is more expressive: you can compose tool calls, loop, branch, save intermediate values, all in one turn.

Wang et al. (2024) showed code-actions outperform JSON tool calls on multi-step reasoning benchmarks.

## Canonical code (with `smolagents` installed)

```python
from smolagents import CodeAgent, LiteLLMModel, tool
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from task import search_corpus, fetch_paper, cite

@tool
def search(query: str) -> list[dict]:
    '''Search the corpus and return top 2 hits.'''
    return search_corpus(query, k=2)

@tool
def fetch(arxiv_id: str) -> dict:
    '''Fetch the full paper record.'''
    return fetch_paper(arxiv_id)

@tool
def cite_claim(arxiv_id: str, claim: str) -> dict:
    '''Verify a claim against a paper.'''
    return cite(arxiv_id, claim)

model = LiteLLMModel(model_id='openai/gpt-4o-mini')
agent = CodeAgent(tools=[search, fetch, cite_claim], model=model)

# The model will emit something like:
#   hits = search('RA-MoE')
#   paper = fetch(hits[0]['arxiv_id'])
#   answer = f"[{paper['arxiv_id']}] {paper['abstract'][:200]}"
#   verified = cite_claim(paper['arxiv_id'], answer)
#   final_answer(answer if verified['supported'] else "I don't know")
result = agent.run('What is RA-MoE?')
```

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import get_question
from eval import smolagents_solve
trace = smolagents_solve(get_question('q01'))
for step in trace.steps:
    print(f"{step.role}: {step.name or ''}  {step.output_summary or step.content or ''}"[:140])

## Tradeoffs

* **+ One model call can do compositional work** (loop, branch, save intermediate) where JSON-tool agents need one call per step.
* **+ Lightweight library** — small dependency footprint compared to the LangChain stack.
* **− Sandbox required** — production needs a real sandbox (E2B), not `LocalPythonInterpreter`.
* **− Less guardrail tooling** than the OpenAI Agents SDK or CrewAI.